In [1]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

# 1. LOAD THE DATASET
dataset = pd.read_excel('Churn_Modelling.xlsx') # Adjust filename as needed
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values

# 2. SPLIT INTO TRAINING AND TEST SET
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

# 3. FEATURE SCALING
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

# 4. TRAIN THE LOGISTIC REGRESSION MODEL
classifier = LogisticRegression(random_state=0)
classifier.fit(X_train, y_train)

# 5. EVALUATE THE MODEL
y_pred = classifier.predict(X_test)

print(f"Logistic Regression Accuracy: {accuracy_score(y_test, y_pred):.2f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

Logistic Regression Accuracy: 0.78
Confusion Matrix:
 [[1527   68]
 [ 378   27]]


In [3]:
import numpy as np
import gradio as gr

# Reuse the trained model/scaler from Cell 1 when available; otherwise train here.
try:
    classifier
    sc
    dataset
except NameError:
    from sklearn.linear_model import LogisticRegression
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    import pandas as pd

    dataset = pd.read_excel("Churn_Modelling.xlsx")
    X = dataset.iloc[:, :-1].values
    y = dataset.iloc[:, -1].values

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=0
    )
    sc = StandardScaler()
    X_train = sc.fit_transform(X_train)
    classifier = LogisticRegression(random_state=0, max_iter=1000)
    classifier.fit(X_train, y_train)

feature_names = dataset.columns[:-1].tolist()
medians = dataset.iloc[:, :-1].median(numeric_only=True)

def predict_churn(*features):
    values = np.array(features, dtype=float).reshape(1, -1)
    scaled_values = sc.transform(values)
    prediction = int(classifier.predict(scaled_values)[0])
    probability = float(classifier.predict_proba(scaled_values)[0][1])

    label = "Likely to churn" if prediction == 1 else "Not likely to churn"
    return label, f"Churn probability: {probability:.2%}"

with gr.Blocks(title="Logistic Regression Churn App") as demo:
    gr.Markdown("## Muhammad Ahmad FA23-BCE-113")
    gr.Markdown("### Logistic Regression Churn Prediction")

    inputs = []
    for name in feature_names:
        default_value = float(medians[name]) if name in medians else 0.0
        inputs.append(gr.Number(label=name, value=default_value))

    with gr.Row():
        prediction_box = gr.Textbox(label="Prediction")
        probability_box = gr.Textbox(label="Probability")

    predict_btn = gr.Button("Predict")
    predict_btn.click(
        fn=predict_churn,
        inputs=inputs,
        outputs=[prediction_box, probability_box],
    )

demo.launch(share=True, prevent_thread_lock=True)

* Running on local URL:  http://127.0.0.1:7861

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/04/01 12:07:24 [W] [service.go:132] login to server failed: tls: failed to verify certificate: x509: certificate has expired or is not yet valid: current time 2026-04-01T12:07:24+05:00 is after 2026-04-01T07:01:35Z
